# HU-6 · Carga en MySQL y vista de KPIs

**Objetivo:** cargar el CSV final de la HU-5 (limpio + cluster) en MySQL y crear una vista con KPIs por medicamento que sirva de fuente a Power BI.

**Flujo (igual que el lab *Load_csv_parquet_to_MySQL*):**
1. Cargar el CSV de la HU-5
2. Recuperar del Excel de la HU-4 las columnas que necesitan los KPIs y no están en el CSV
3. Elegir las columnas que van a MySQL
4. Conectar con MySQL
5. Crear la tabla *(lo escribes tú)*
6. Insertar las filas y hacer commit
7. Verificar en SQL
8. Crear la vista de KPIs
9. Conclusiones

In [2]:
import pandas as pd
import mysql.connector
from getpass import getpass

RUTA_CLUSTERS = "../datos/medicamentos_artritis_reumatoide_clusters.csv"   # salida de la HU-5
RUTA_HU4 = "../datos/medicamentos_artritis_reumatoide_hu4.xlsx"            # Excel completo de la HU-4

## 1. Cargar el CSV de la HU-5
Es el fichero que pide la HU-6. `nregistro` se lee como texto para no perder los ceros a la izquierda.

In [3]:
df_clu = pd.read_csv(RUTA_CLUSTERS, dtype={"nregistro": str})
print(df_clu.shape)
print(df_clu.columns.tolist())

(645, 34)
['nregistro', 'nombre', 'volumen_informativo_seguridad', 'complejidad_desglose_clinico', 'indicador_riesgo_severo', 'pvp_iva', 'dosis_mg', 'antiguedad_anios', 'num_principios_activos', 'comercializado', 'generico', 'biosimilar', 'afecta_conduccion', 'triangulo_negro', 'tratamiento_larga_duracion', 'tiene_precio', 'tiene_revision', 'inyectable', 'estado_alta', 'estado_baja', 'via_intraarticular', 'via_intramuscular', 'via_intravenosa', 'via_oral', 'via_rectal', 'via_subcutanea', 'via_otras', 'forma_inyectable', 'forma_oral_liquido', 'forma_oral_solido', 'forma_perfusion', 'forma_rectal', 'cluster', 'nombre_cluster']


## 2. Recuperar las columnas que faltan
El CSV de la HU-5 no trae `cn`, `pactivos`, `precio_referencia`, `num_excipientes` ni `forma_farmaceutica_simplificada`, porque se quitaron en la HU-4/HU-5. Sí están en el Excel de la HU-4, así que se añaden uniendo por `nregistro`.

- **Por qué aquí y no cambiando la HU-5:** la HU-5 está cerrada; juntar fuentes antes de cargar es trabajo de la fase de carga.
- **`how="left"`:** *(explícalo tú)*
- **`validate="1:1"`:** *(explícalo tú)*

In [4]:
df_hu4 = pd.read_excel(RUTA_HU4, dtype={"nregistro": str, "cn": str})

# Solo se traen las columnas que el CSV no tenga ya (si no, pandas crea sufijos _x/_y)
cols_extra = ["cn", "pactivos", "precio_referencia", "num_excipientes", "forma_farmaceutica_simplificada",
              "precio_rellenado"]
cols_faltan = [c for c in cols_extra if c not in df_clu.columns]
print("Se recuperan:", cols_faltan)

df = df_clu.merge(df_hu4[["nregistro"] + cols_faltan], on="nregistro", how="left", validate="1:1")

print(df_clu.shape, "->", df.shape)
print(df[cols_extra].isna().sum())
print((df["pvp_iva"] == df["precio_referencia"]).sum(), "filas con PVP == PR")

Se recuperan: ['cn', 'pactivos', 'precio_referencia', 'num_excipientes', 'forma_farmaceutica_simplificada', 'precio_rellenado']
(645, 34) -> (645, 40)
cn                                   0
pactivos                             0
precio_referencia                  329
num_excipientes                      0
forma_farmaceutica_simplificada      0
precio_rellenado                     0
dtype: int64
316 filas con PVP == PR


### 2.1 Quitar los precios inventados
En la HU-4 del grupo se rellenó el precio de **150 medicamentos que no están en el Nomenclátor** (`estado_nomenclator = "SIN DATO"`, marcados con `precio_rellenado = 1`). Para el K-Means era una decisión válida, pero un KPI de negocio no puede calcularse con precios inventados: Power BI mostraría desviaciones, márgenes y rankings falsos.

→ En la tabla de MySQL esos precios pasan a `NULL` y `tiene_precio` se recalcula. Los KPIs de precio saldrán `NULL` para ellos, que es la respuesta honesta: no sabemos su precio.

In [5]:
print("Precios rellenados:", (df["precio_rellenado"] == 1).sum())
df.loc[df["precio_rellenado"] == 1, "pvp_iva"] = None
df["tiene_precio"] = df["pvp_iva"].notna().astype(int)

print("pvp_iva nulos ahora:", df["pvp_iva"].isna().sum())   # deben ser 306 = SIN DATO
print(pd.crosstab(df_hu4.set_index("nregistro").loc[df["nregistro"], "estado_nomenclator"].values,
                  df["pvp_iva"].isna(), rownames=["estado_nomenclator"], colnames=["pvp nulo"]))

Precios rellenados: 150
pvp_iva nulos ahora: 306
pvp nulo            False  True 
estado_nomenclator              
ALTA                  247      0
BAJA                   92      0
SIN DATO                0    306


## 3. Elegir las columnas que van a MySQL
Solo las necesarias para calcular los KPIs y para filtrar en Power BI. Las columnas 0/1 de vías y formas repiten lo que ya dice `forma_farmaceutica_simplificada`.

In [6]:
columnas = ["nregistro", "cn", "nombre", "pactivos", "forma_farmaceutica_simplificada", "dosis_mg",
            "cluster", "nombre_cluster", "pvp_iva", "precio_referencia", "tiene_precio",
            "volumen_informativo_seguridad", "complejidad_desglose_clinico", "indicador_riesgo_severo",
            "num_principios_activos", "num_excipientes", "comercializado", "generico", "biosimilar",
            "antiguedad_anios"]
df_load = df[columnas].copy()

print(df_load.shape)
print(df_load.isna().sum()[lambda s: s > 0])
print("nregistro único:", df_load["nregistro"].is_unique, "| cn único:", df_load["cn"].is_unique)

(645, 20)
dosis_mg               5
pvp_iva              306
precio_referencia    329
dtype: int64
nregistro único: True | cn único: True


## 4. Conectar con MySQL
Con el usuario `arthena_app` (solo tiene permisos sobre la base `arthena`). La contraseña se pide con `getpass` para que no quede escrita en el notebook ni en GitHub.

In [7]:
conn = mysql.connector.connect(host="localhost", port=3306, user="arthena_app",
                               password=getpass("Clave MySQL:"), database="arthena")
print("Conectado:", conn.is_connected())

Conectado: True


## 5. Crear la tabla *(lo escribes tú)*
Antes de escribir el `CREATE TABLE`, decide y justifica:
- ¿`nregistro` y `cn` como `INT` o `VARCHAR`?
- ¿Precios como `FLOAT` o `DECIMAL(10,2)`?
- ¿Qué columna es la clave primaria?
- ¿Qué columnas pueden ser `NULL`?

In [8]:
print(df_load.dtypes)
for col in ["nregistro", "cn", "nombre", "pactivos", "forma_farmaceutica_simplificada", "nombre_cluster"]:
    print(col, "longitud máx:", df_load[col].astype(str).str.len().max())

nregistro                              str
cn                                     str
nombre                                 str
pactivos                               str
forma_farmaceutica_simplificada        str
dosis_mg                           float64
cluster                              int64
nombre_cluster                         str
pvp_iva                            float64
precio_referencia                  float64
tiene_precio                         int64
volumen_informativo_seguridad        int64
complejidad_desglose_clinico         int64
indicador_riesgo_severo              int64
num_principios_activos               int64
num_excipientes                      int64
comercializado                       int64
generico                             int64
biosimilar                           int64
antiguedad_anios                   float64
dtype: object
nregistro longitud máx: 10
cn longitud máx: 6
nombre longitud máx: 93
pactivos longitud máx: 68
forma_farmaceutica_simplificad

In [10]:
# CREATE TABLE
sql_tabla = """
CREATE TABLE medicamentos (
    nregistro                        VARCHAR(10)   NOT NULL,
    cn                               VARCHAR(10)   NOT NULL,
    nombre                           VARCHAR(300)  NOT NULL,
    pactivos                         VARCHAR(500),
    forma_farmaceutica_simplificada  VARCHAR(100),
    dosis_mg                         DECIMAL(10,2),
    cluster                          TINYINT       NOT NULL,
    nombre_cluster                   VARCHAR(100)  NOT NULL,
    pvp_iva                          DECIMAL(10,2),
    precio_referencia                DECIMAL(10,2),
    tiene_precio                     TINYINT(1)    NOT NULL,
    volumen_informativo_seguridad    INT,
    complejidad_desglose_clinico     INT,
    indicador_riesgo_severo          INT,
    num_principios_activos           TINYINT,
    num_excipientes                  TINYINT,
    comercializado                   TINYINT(1),
    generico                         TINYINT(1),
    biosimilar                       TINYINT(1),
    antiguedad_anios                 DECIMAL(5,1),
    PRIMARY KEY (nregistro)
)"""
cursor = conn.cursor()
cursor.execute("DROP TABLE IF EXISTS medicamentos")
cursor.execute(sql_tabla)
cursor.execute("DESCRIBE medicamentos")
for fila in cursor.fetchall(): print(fila)

('nregistro', 'varchar(10)', 'NO', 'PRI', None, '')
('cn', 'varchar(10)', 'NO', '', None, '')
('nombre', 'varchar(300)', 'NO', '', None, '')
('pactivos', 'varchar(500)', 'YES', '', None, '')
('forma_farmaceutica_simplificada', 'varchar(100)', 'YES', '', None, '')
('dosis_mg', 'decimal(10,2)', 'YES', '', None, '')
('cluster', 'tinyint', 'NO', '', None, '')
('nombre_cluster', 'varchar(100)', 'NO', '', None, '')
('pvp_iva', 'decimal(10,2)', 'YES', '', None, '')
('precio_referencia', 'decimal(10,2)', 'YES', '', None, '')
('tiene_precio', 'tinyint(1)', 'NO', '', None, '')
('volumen_informativo_seguridad', 'int', 'YES', '', None, '')
('complejidad_desglose_clinico', 'int', 'YES', '', None, '')
('indicador_riesgo_severo', 'int', 'YES', '', None, '')
('num_principios_activos', 'tinyint', 'YES', '', None, '')
('num_excipientes', 'tinyint', 'YES', '', None, '')
('comercializado', 'tinyint(1)', 'YES', '', None, '')
('generico', 'tinyint(1)', 'YES', '', None, '')
('biosimilar', 'tinyint(1)', 'YES'

## 6. Insertar las filas y hacer commit

In [ ]:
df_sql = df_load.astype(object).where(df_load.notna(), None)
filas = list(df_sql.itertuples(index=False, name=None))

marcadores = ", ".join(["%s"] * len(columnas))
sql_insert = f"INSERT INTO medicamentos ({', '.join(columnas)}) VALUES ({marcadores})"

cursor.execute("DELETE FROM medicamentos")
cursor.executemany(sql_insert, filas)

cursor.execute("SELECT COUNT(*) FROM medicamentos")
n_mysql = cursor.fetchone()[0]
print("Filas en pandas:", len(df_load), "| filas en MySQL:", n_mysql)

if n_mysql == len(df_load):
    conn.commit(); print("COMMIT hecho")
else:
    conn.rollback(); print("Los conteos no cuadran: ROLLBACK")

Filas en pandas: 645 | filas en MySQL: 645
COMMIT hecho


[('61836',
  '664706',
  'ACABEL 4 mg COMPRIMIDOS RECUBIERTOS CON PELICULA',
  'LORNOXICAM',
  'COMPRIMIDO',
  4.0,
  1,
  'AINE genéricos económicos',
  2.5,
  None,
  1,
  1318,
  1,
  6,
  1,
  0,
  0,
  0,
  0,
  28.5),
 ('61837',
  '664698',
  'ACABEL 8 mg COMPRIMIDOS RECUBIERTOS CON PELICULA',
  'LORNOXICAM',
  'COMPRIMIDO',
  8.0,
  1,
  'AINE genéricos económicos',
  4.51,
  None,
  1,
  1319,
  1,
  6,
  1,
  0,
  0,
  0,
  0,
  28.5),
 ('69223',
  '659777',
  'ACECLOFENACO CINFA 100 mg COMPRIMIDOS RECUBIERTOS CON PELICULA EFG',
  'ACECLOFENACO',
  'COMPRIMIDO',
  100.0,
  1,
  'AINE genéricos económicos',
  2.83,
  2.83,
  1,
  1127,
  1,
  8,
  1,
  0,
  1,
  1,
  0,
  19.2),
 ('71437',
  '663991',
  'ACECLOFENACO NORMON 100 mg COMPRIMIDOS RECUBIERTOS CON PELICULA EFG',
  'ACECLOFENACO',
  'COMPRIMIDO',
  100.0,
  1,
  'AINE genéricos económicos',
  2.83,
  2.83,
  1,
  1124,
  1,
  8,
  1,
  0,
  1,
  1,
  0,
  17.1),
 ('79696',
  '706012',
  'ACECLOFENACO STADA 100 MG COMP

## 7. Verificar en SQL

## 8. Vista de KPIs

## 9. Conclusiones